# Part B — Matrices

**Final Project: The Linear Algebra Behind AI — A Regression Problem**

Your entire dataset can be written as a single matrix — this is the object every later step operates on.

---
**Student name:** _______________________   **Date:** _______________________

---


**What to do:** Represent your dataset as a design matrix X ∈ ℝⁿˣᵖ. Compute the product XᵀX.

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
import os, numpy as np
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data = np.load('/content/drive/MyDrive/partA_data.npz', allow_pickle=True)
    X_raw, y = data['X_raw'], data['y']
    feature_names = data['feature_names'].tolist()
except Exception:
    if os.path.exists('partA_data.npz'):
        data = np.load('partA_data.npz', allow_pickle=True)
        X_raw, y = data['X_raw'], data['y']
        feature_names = data['feature_names'].tolist()
    else:
        get_ipython().run_line_magic('store', '-r X_raw y feature_names')
print('Loaded X_raw:', X_raw.shape, ' y:', y.shape)

Loaded X_raw: (6607, 4)  y: (6607,)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [3]:
# Build the design matrix X (rows = samples, columns = features).
# X_raw is already a clean numeric matrix (4 numeric features, no missing values),
# so we only need to make sure it is a float array.
X = np.asarray(X_raw, dtype=float)
n, p = X.shape
print("X shape:", X.shape)
print(f"n = {n} samples, p = {p} features: {feature_names}")

X shape: (6607, 4)
n = 6607 samples, p = 4 features: ['Attendance', 'Hours_Studied', 'Previous_Scores', 'Sleep_Hours']


In [4]:
# Compute the matrix product X^T X  (p x n) @ (n x p) -> (p x p)
XtX = X.T @ X
print("X^T X shape:", XtX.shape)
print(XtX)
print("Symmetric?", np.allclose(XtX, XtX.T))

X^T X shape: (4, 4)
[[43141835. 10550656. 39645921.  3712450.]
 [10550656.  2873355.  9921742.   928312.]
 [39645921.  9921742. 38604087.  3483313.]
 [ 3712450.   928312.  3483313.   340675.]]
Symmetric? True


In [5]:
# Inspect one off-diagonal entry of X^T X (pick any i != j)
i, j = 0, 1   # Attendance and Hours_Studied
print(f"(X^T X)[{i},{j}] = {XtX[i, j]:.4f}  -- corresponds to features "
      f"'{feature_names[i]}' and '{feature_names[j]}'")

# Check: it equals the dot product of the two columns
print("Dot product of the two columns:", X[:, i] @ X[:, j])

(X^T X)[0,1] = 10550656.0000  -- corresponds to features 'Attendance' and 'Hours_Studied'
Dot product of the two columns: 10550656.0


### Answer these in your report

**Q5.** What are n and p for your dataset? What does each row of X represent? What does each column represent?

> *Your answer:* The dataset has n = 6,607 samples and p = 4 features, so X is a 6,607 × 4 matrix (X ∈ ℝ⁶⁶⁰⁷ˣ⁴). Each row represents one student. Each column represents one predictor measured for every student: Attendance (percentage of classes attended), Hours_Studied (weekly study hours), Previous_Scores (score on earlier exams), and Sleep_Hours (average hours of sleep per night). The target Exam_Score is kept separately as the vector y and is not part of X.

**Q6.** What are the dimensions of XᵀX, and why is that shape useful for the steps that come later?

> *Your answer:* XᵀX is p × p, so here it is 4 × 4, because (p × n)(n × p) = p × p. Its size depends only on the number of features, not on the number of students, so even with 6,607 rows we only have to work with a tiny 4 × 4 matrix. It is also square and symmetric, which means we can ask whether it is invertible, which is needed for the normal equations β = (XᵀX)⁻¹Xᵀy in least squares, and we can compute its eigenvalues, which is needed for PCA and for checking conditioning.

**Q7.** Pick one single off-diagonal entry of XᵀX and explain, in words, what it measures about your two corresponding features.

> *Your answer:* I picked `(XᵀX)[0,1] = 10,550,656.0000`, which is the dot product of the `Attendance` column and the `Hours_Studied` column: the sum over all $6,607$ students of $(\text{attendance} \times \text{hours studied})$. It measures how much the two features line up: it is large when students who have high attendance also tend to have high study hours, and it is small when they do not. Because the features are not centered or standardized yet, the number is also inflated by their overall size (all values are positive), so it is not a correlation. Once the columns are centered and scaled, this same entry (divided by $n - 1$) becomes the covariance or correlation between the two features.

**Q8.** If you added one new sample to your dataset, how would X change? Would XᵀX change in the same simple way? Explain.

> *Your answer:* X would gain one new row xₙ₊₁ᵀ (a 1 × 4 row of that student's features), so it would go from n × p to (n+1) × p. XᵀX would stay p × p, and it would change in a simple additive way: X′ᵀX′ = XᵀX + xₙ₊₁xₙ₊₁ᵀ, where xₙ₊₁xₙ₊₁ᵀ is a rank-one p × p matrix. This holds because XᵀX is a sum of outer products, one per student (XᵀX = Σₖ xₖxₖᵀ), so a new student just adds one more term. The change to X is stacking a row, and the change to XᵀX is adding a matrix; they are not the same kind of change.

### Save your work for the next notebook

In [6]:
%store X
%store XtX
np.savez('partB_data.npz', X=X, XtX=XtX, y=y, feature_names=feature_names)
try:
    from google.colab import drive
    if os.path.exists('/content/drive/MyDrive'):
        np.savez('/content/drive/MyDrive/partB_data.npz', X=X, XtX=XtX, y=y, feature_names=feature_names)
except Exception:
    pass
print("Saved X, XtX for Part C.")

Saved X, XtX for Part C.
